In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.collections import LineCollection
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import cm
from scipy.interpolate import interp1d
from scipy.stats import spearmanr, linregress

SHP_PATH       = Path("shape/Flowlines_huc02_12_Brazos.shp")
RESULT_DIR     = Path("outputs_flood_rl2")
FIG_DIR        = Path("figures_wrr_new"); FIG_DIR.mkdir(parents=True, exist_ok=True)

GCMS = ["MPI-ESM1-2-HR", "ACCESS-CM2", "EC-Earth3",
        "CNRM-ESM2-1", "BCC-CSM2-MR", "MRI-ESM2-0", "NorESM2-MM"]
SCENARIOS     = [126, 245, 370, 585]
REF           = 126
TARGET_BUDGET = 10.0
BUDGET_TOL    = 0.10
SIGMA_MS      = 12

SSP_LABELS = {126:"SSP1-2.6", 245:"SSP2-4.5", 370:"SSP3-7.0", 585:"SSP5-8.5"}
SSP_COLORS = {126:"#2d8e3d", 245:"#2166AC", 370:"#d4820a", 585:"#c72e29"}
SSP_COLORS_LIGHT = {126:"#a8ddb5", 245:"#9ecae1", 370:"#fdd49e", 585:"#fcae91"}

MM = 1/25.4; W_FULL = 190*MM; W_HALF = 95*MM

mpl.rcParams.update({
    "font.family": "Arial", "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7, "axes.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 6, "ytick.labelsize": 6,
    "xtick.major.width": 0.4, "ytick.major.width": 0.4,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.fontsize": 6, "legend.frameon": False,
    "legend.handlelength": 1.2, "legend.handletextpad": 0.4,
    "figure.dpi": 200, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42, "lines.linewidth": 0.8,
})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INTERPOLATION HELPER
# ═══════════════════════════════════════════════════════════════════
def interpolate_to_budget(grp, target=TARGET_BUDGET):
    eff_by_r = grp.groupby("R_weight")["eff_total_L1"].sum() / 1e9
    eff_by_r = eff_by_r.sort_index()
    r_values = eff_by_r.index.values
    efforts  = eff_by_r.values

    if len(r_values) < 2:
        return grp[grp["R_weight"] == r_values[0]].copy()

    sort_idx = np.argsort(efforts)
    efforts_sorted = efforts[sort_idx]
    r_sorted       = r_values[sort_idx]

    if target <= efforts_sorted[0]:
        return grp[grp["R_weight"] == r_sorted[0]].copy()
    elif target >= efforts_sorted[-1]:
        return grp[grp["R_weight"] == r_sorted[-1]].copy()

    idx_above = np.searchsorted(efforts_sorted, target)
    idx_below = idx_above - 1
    eff_lo, eff_hi = efforts_sorted[idx_below], efforts_sorted[idx_above]
    r_lo, r_hi     = r_sorted[idx_below], r_sorted[idx_above]
    w = 0.5 if eff_hi == eff_lo else (target - eff_lo) / (eff_hi - eff_lo)

    df_lo = grp[grp["R_weight"] == r_lo].set_index("reach_id")
    df_hi = grp[grp["R_weight"] == r_hi].set_index("reach_id")
    common = df_lo.index.intersection(df_hi.index)
    df_lo, df_hi = df_lo.loc[common], df_hi.loc[common]

    result = df_lo.copy()
    for col in df_lo.select_dtypes(include=[np.number]).columns:
        if col in ("R_weight", "scenario", "reach_id"): continue
        result[col] = df_lo[col] * (1 - w) + df_hi[col] * w
    return result.reset_index()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════
print("Loading spatial data...")
gdf_net = gpd.read_file(SHP_PATH)
gdf_net["COMID"] = gdf_net["COMID"].astype(int)

frames = []
for gcm in GCMS:
    for pattern in [f"results_*_{gcm}.parquet", f"results_{gcm}.parquet"]:
        for fp in RESULT_DIR.glob(pattern):
            frames.append(pd.read_parquet(fp))
df_all = pd.concat(frames, ignore_index=True)
df_all["reach_id"] = df_all["reach_id"].astype(int)
df_all["scenario"] = df_all["scenario"].astype(int)
if "hydro" not in df_all.columns:
    df_all["hydro"] = "VIC5"
    print("  ⚠ No 'hydro' column — assuming single hydro model")

print(f"  Interpolating to budget = {TARGET_BUDGET} Gm³/yr ...")
sel = []
for (gcm, scn, hydro), grp in df_all.groupby(["gcm", "scenario", "hydro"]):
    interp = interpolate_to_budget(grp, TARGET_BUDGET)
    sel.append(interp)
    actual_eff = interp["eff_total_L1"].sum() / 1e9
    print(f"    {hydro:>5s} {gcm:<18s} SSP{scn}: effort={actual_eff:.1f} Gm³/yr")

df_sel = pd.concat(sel, ignore_index=True)
df_sel["scenario"] = df_sel["scenario"].astype(int)
df_sel["reach_id"] = df_sel["reach_id"].astype(int)

mcols = [c for c in ["vol_hi_baseline", "vol_hi_controlled", "vol_hi_red_rel",
                      "peak_baseline", "peak_controlled", "peak_red_rel",
                      "events_baseline", "eff_total_L1", "residual_ratio"]
         if c in df_sel.columns]
df_ens = df_sel.groupby(["reach_id", "scenario"])[mcols].median().reset_index()

# ── Merge network attributes into df_ens ─────────────────────────
df_ens = df_ens.merge(
    gdf_net[["COMID", "StreamOrde", "TotDASqKM"]].rename(
        columns={"COMID": "reach_id"}),
    on="reach_id", how="left")

if "mean_flow_ref" in df_sel.columns:
    df_ens["mean_flow_ref"] = df_ens["reach_id"].map(
        df_sel.groupby("reach_id")["mean_flow_ref"].first())

# ── Derived columns ──────────────────────────────────────────────
df_ens["vol_hi_red_abs"] = (df_ens["vol_hi_baseline"]
                            - df_ens["vol_hi_controlled"])
df_ens["eff_norm_baseline"] = (df_ens["eff_total_L1"]
                               / df_ens["vol_hi_baseline"].clip(lower=1e-6))

gdf_by = {}
for scn in SCENARIOS:
    gdf_by[scn] = gdf_net.merge(df_ens[df_ens["scenario"] == scn],
                                 left_on="COMID", right_on="reach_id",
                                 how="inner")

n_ens   = df_sel.groupby(["gcm", "hydro"]).ngroups
n_gcm   = df_sel["gcm"].nunique()
n_hydro = df_sel["hydro"].nunique()
print(f"\n  Reaches: {df_ens['reach_id'].nunique():,}, "
      f"Ensemble: {n_ens} ({n_gcm} GCMs × {n_hydro} hydro), "
      f"Scenarios: {df_sel['scenario'].nunique()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ENSEMBLE AGGREGATION
# ═══════════════════════════════════════════════════════════════════
print("\nComputing ensemble aggregation ...")
RETURN_PERIODS_AGG = [2, 5, 10, 20, 50, 100]

_gcm_group_cols = ["gcm", "scenario"]
if "hydro" in df_sel.columns:
    _gcm_group_cols = ["gcm", "hydro", "scenario"]

_agg_dict = {
    "vol_hi_baseline":   [("vol_baseline_Gm3yr",   lambda x: x.sum() / 1e9)],
    "vol_hi_controlled": [("vol_controlled_Gm3yr", lambda x: x.sum() / 1e9)],
    "eff_total_L1":      [("effort_Gm3yr",         lambda x: x.sum() / 1e9)],
    "peak_baseline":     [("peak_baseline_sum", "sum"),
                          ("peak_baseline_med", "median")],
    "peak_controlled":   [("peak_controlled_sum", "sum"),
                          ("peak_controlled_med", "median")],
    "peak_red_rel":      [("peak_red_rel_med", "median")],
    "events_baseline":   [("events_baseline_sum", "sum")],
    "events_controlled": [("events_controlled_sum", "sum")],
    "events_red_rel":    [("events_red_rel_med", "median")],
}
for T in RETURN_PERIODS_AGG:
    for suffix, agg in [("baseline_sum", "sum"), ("baseline_med", "median"),
                        ("controlled_sum", "sum"), ("controlled_med", "median")]:
        col = f"rl{T}_{'baseline' if 'baseline' in suffix else 'controlled'}"
        out = f"rl{T}_{suffix}"
        if col in df_sel.columns:
            _agg_dict.setdefault(col, []).append((out, agg))
    rd = f"rl{T}_red_rel"
    if rd in df_sel.columns:
        _agg_dict.setdefault(rd, []).append((f"rl{T}_red_rel_med", "median"))
if "nse_openloop" in df_sel.columns:
    _agg_dict["nse_openloop"] = [("nse_openloop_med", "median")]

_named_agg = {}
for src_col, agg_list in _agg_dict.items():
    if src_col not in df_sel.columns: continue
    for out_name, func in agg_list:
        _named_agg[out_name] = pd.NamedAgg(column=src_col, aggfunc=func)

ens_gcm = (df_sel.groupby(_gcm_group_cols, as_index=False).agg(**_named_agg))
ens_gcm["budget_x"] = TARGET_BUDGET
ens_gcm["vol_reduced_Gm3yr"] = ens_gcm["vol_baseline_Gm3yr"] - ens_gcm["vol_controlled_Gm3yr"]
ens_gcm["vol_red_rel_basin"] = ens_gcm["vol_reduced_Gm3yr"] / (ens_gcm["vol_baseline_Gm3yr"] + 1e-12)

_metric_cols = [c for c in ens_gcm.columns
                if c not in ("gcm", "hydro", "scenario", "budget_x")]
_ens_rows = []
for scn, grp in ens_gcm.groupby("scenario"):
    row = {"scenario": scn, "budget_x": TARGET_BUDGET, "n_members": len(grp)}
    for col in _metric_cols:
        vals = grp[col].dropna()
        if len(vals) == 0: continue
        row[f"{col}_gcm_med"] = vals.median()
        row[f"{col}_gcm_q25"] = vals.quantile(0.25)
        row[f"{col}_gcm_q75"] = vals.quantile(0.75)
    _ens_rows.append(row)
ens_summ = pd.DataFrame(_ens_rows)
has_ens = True
print(f"  ens_gcm: {len(ens_gcm)} rows, ens_summ: {len(ens_summ)} rows")

# ── Daily & FDC data (for Fig 2) ────────────────────────────────
daily_frames = [pd.read_parquet(f) for f in sorted(RESULT_DIR.glob("daily_*.parquet"))]
df_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else None
if df_daily is not None:
    df_daily = df_daily[df_daily["budget_target"] == TARGET_BUDGET]
    df_daily["date"] = pd.to_datetime(df_daily["date"])
    df_daily["year"] = df_daily["date"].dt.year
    print(f"  Daily data: {len(df_daily):,} rows")

fdc_frames = [pd.read_parquet(f) for f in sorted(RESULT_DIR.glob("fdc_*.parquet"))]
df_fdc = pd.concat(fdc_frames, ignore_index=True) if fdc_frames else None
if df_fdc is not None:
    df_fdc = df_fdc[df_fdc["budget_target"] == TARGET_BUDGET]
    print(f"  FDC data: {len(df_fdc):,} rows")

val_cols = [c for c in ["nse_openloop", "nse_anchored", "pbias_openloop",
                         "rl10_bias_openloop"] if c in df_sel.columns]
print(f"  Validation columns: {val_cols}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════
def _segs(geom):
    if geom is None or geom.is_empty: return []
    if geom.geom_type == "LineString": return [np.asarray(geom.coords)]
    if geom.geom_type == "MultiLineString":
        return [np.asarray(g.coords) for g in geom.geoms]
    return []


def draw_net_cmap(ax, gdf, values, cmap, norm, lw, alpha=0.92):
    segs, cols, lws = [], [], []
    for geom, v, w in zip(gdf.geometry.values, values, lw):
        for part in _segs(geom):
            segs.append(part)
            cols.append(cmap(norm(v)) if np.isfinite(v) else (0.85,0.85,0.85,0.3))
            lws.append(float(w))
    lc = LineCollection(segs, colors=cols, linewidths=lws,
                        capstyle="round", joinstyle="round", alpha=alpha, zorder=1)
    lc.set_rasterized(True); ax.add_collection(lc)


def draw_net_rgba(ax, gdf, colors, lw, alpha=0.92):
    segs, cols, lws = [], [], []
    for geom, c, w in zip(gdf.geometry.values, colors, lw):
        for part in _segs(geom):
            segs.append(part); cols.append(c); lws.append(float(w))
    lc = LineCollection(segs, colors=cols, linewidths=lws,
                        capstyle="round", joinstyle="round", alpha=alpha, zorder=1)
    lc.set_rasterized(True); ax.add_collection(lc)


def _flow_weighted_lw(gdf, lo=0.15, hi=3.5):
    da = np.clip(gdf["TotDASqKM"].to_numpy(float), 1, None)
    ld = np.log10(da); mn, mx = ld.min(), ld.max()
    if mx <= mn: return np.full(len(da), (lo+hi)/2)
    return lo + (hi-lo) * (ld - mn) / (mx - mn)


def lw_order(gdf, lo=0.3, hi=3.5):
    so = gdf["StreamOrde"].to_numpy(float)
    return lo + (hi - lo) * (so / max(float(np.nanmax(so)), 1.0))


def add_bg(ax, gdf):
    gdf.plot(ax=ax, color="#f0f0f0", linewidth=0.02, alpha=0.22, zorder=0)


def add_scalebar(ax, gdf, km=100):
    b = gdf.total_bounds; bar = km / 96.0
    x0 = b[2] - bar - (b[2]-b[0])*0.04
    y0 = b[1] + (b[3]-b[1])*0.04
    ax.plot([x0, x0+bar], [y0, y0], "k-", lw=0.8, solid_capstyle="butt", zorder=10)
    ax.text(x0+bar/2, y0+(b[3]-b[1])*0.015, f"{km} km",
            ha="center", fontsize=5, zorder=10)


def plabel(ax, lab, x=-0.08, y=1.05):
    ax.text(x, y, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="right")


def plabel_map(ax, lab):
    ax.text(0.03, 0.97, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9))


def tcmap(name, lo=0.0, hi=1.0, n=256):
    base = cm.get_cmap(name)
    return mcolors.LinearSegmentedColormap.from_list(
        f"{name}_t", base(np.linspace(lo, hi, n)))


def lgrid(ax, alpha=0.10, lw=0.3):
    ax.grid(True, alpha=alpha, lw=lw, zorder=0); ax.set_axisbelow(True)


def make_colorbar(fig, pos, cmap, norm, label, ticks=None, ticklabels=None):
    cax = fig.add_axes(pos)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label(label, fontsize=6)
    if ticks is not None: cb.set_ticks(ticks)
    if ticklabels is not None: cb.set_ticklabels(ticklabels)
    cb.ax.tick_params(labelsize=5.5, width=0.3, length=2)
    cb.outline.set_linewidth(0.3)


def make_agreement_cmap(n_members):
    blue_to_red = cm.get_cmap("RdBu_r")
    colors = [blue_to_red(i / n_members) for i in range(n_members + 1)]
    cmap = mcolors.ListedColormap(colors)
    bounds = [i - 0.5 for i in range(n_members + 2)]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    return cmap, norm


def _weighted_quantile(values, weights, q):
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    v, w = values[mask], weights[mask]
    if len(v) == 0: return np.nan
    si = np.argsort(v); v, w = v[si], w[si]
    cumw = np.cumsum(w); cumw = cumw / cumw[-1]
    return float(np.interp(q, cumw, v))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 2 — Time Series + Flow Duration Curve
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 2 — Time series + FDC")
print("=" * 60)

if df_daily is not None and df_fdc is not None:
    fig = plt.figure(figsize=(W_FULL, 100*MM))
    gs_left = fig.add_gridspec(2, 2, left=0.08, right=0.54,
                                top=0.95, bottom=0.12, hspace=0.12, wspace=0.15)
    gs_right = fig.add_gridspec(1, 1, left=0.63, right=0.98,
                                 top=0.95, bottom=0.12)

    y_min, y_max = 0.1, 5000
    panel_pos = [(0, 0), (0, 1), (1, 0), (1, 1)]

    for idx, scn in enumerate(SCENARIOS):
        row, col = panel_pos[idx]
        ax = fig.add_subplot(gs_left[row, col])
        scn_data = df_daily[df_daily["scenario"] == scn]

        grp_cols = ["gcm", "year"]
        if "hydro" in scn_data.columns: grp_cols = ["gcm", "hydro", "year"]
        annual = scn_data.groupby(grp_cols).agg(
            bl=("basin_exceed_baseline", "sum"),
            cl=("basin_exceed_controlled", "sum")).reset_index()
        annual["bl_Gm3"] = annual["bl"] / 1e9
        annual["cl_Gm3"] = annual["cl"] / 1e9

        ens_bl = annual.groupby("year")["bl_Gm3"].agg(
            med="median", q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75)).reset_index()
        ens_cl = annual.groupby("year")["cl_Gm3"].agg(
            med="median", q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75)).reset_index()

        ax.fill_between(ens_bl["year"], ens_bl["q25"], ens_bl["q75"],
                        color="#ccc", alpha=0.5, lw=0)
        ax.plot(ens_bl["year"], ens_bl["med"], color="#666", ls="--", lw=0.8)
        ax.fill_between(ens_cl["year"], ens_cl["q25"], ens_cl["q75"],
                        color=SSP_COLORS[scn], alpha=0.3, lw=0)
        ax.plot(ens_cl["year"], ens_cl["med"], color=SSP_COLORS[scn], lw=1.0)

        ax.set_yscale("log"); ax.set_ylim(y_min, y_max)
        ax.set_xlim(2020, 2100); ax.set_xticks([2020, 2060, 2100])
        ax.yaxis.grid(True, which="major", color="#e0e0e0", lw=0.4)
        ax.set_axisbelow(True)
        ax.text(0.97, 0.93, SSP_LABELS[scn], transform=ax.transAxes,
                fontsize=7, fontweight="bold", color=SSP_COLORS[scn],
                ha="right", va="top")
        plabel(ax, chr(ord("a") + idx), x=-0.02, y=1.02)
        if row == 0: ax.set_xticklabels([])
        if col == 1: ax.set_yticklabels([])
        if idx == 1:
            leg = [Line2D([0],[0], color="#666", ls="--", lw=0.8, label="Baseline"),
                   Line2D([0],[0], color="#666", lw=1.0, label="Controlled")]
            ax.legend(handles=leg, loc="lower right", fontsize=5, handlelength=1.2)

    fig.text(0.02, 0.54, r"Exceedance above $Q_{RL2}$ (Gm$^3$/yr)",
             va="center", ha="center", rotation="vertical", fontsize=7)

    ax_fdc = fig.add_subplot(gs_right[0, 0])
    pct_cols = [c for c in df_fdc.columns if c.startswith("fdc_") and "baseline" in c]
    percentiles = sorted([int(c.split("_")[1].replace("pct","")) for c in pct_cols])
    for scn in SCENARIOS:
        sd = df_fdc[df_fdc["scenario"] == scn]
        bl_med, bl_q25, bl_q75 = [], [], []
        cl_med, cl_q25, cl_q75 = [], [], []
        for p in percentiles:
            bv = sd[f"fdc_{p}pct_baseline"] / 1e5
            bl_med.append(bv.median()); bl_q25.append(bv.quantile(0.25)); bl_q75.append(bv.quantile(0.75))
            cv = sd[f"fdc_{p}pct_controlled"] / 1e5
            cl_med.append(cv.median()); cl_q25.append(cv.quantile(0.25)); cl_q75.append(cv.quantile(0.75))
        ax_fdc.fill_between(percentiles, bl_q25, bl_q75, color=SSP_COLORS[scn], alpha=0.08, lw=0)
        ax_fdc.plot(percentiles, bl_med, color=SSP_COLORS[scn], ls="--", lw=0.8)
        ax_fdc.fill_between(percentiles, cl_q25, cl_q75, color=SSP_COLORS[scn], alpha=0.15, lw=0)
        ax_fdc.plot(percentiles, cl_med, color=SSP_COLORS[scn], lw=1.4)

    ax_fdc.set_xlabel("Exceedance probability (%)")
    ax_fdc.set_ylabel(r"Basin total flow ($\times 10^5$ m$^3$/s)")
    ax_fdc.set_xscale("log"); ax_fdc.set_xlim(1, 100)
    ax_fdc.set_xticks([1, 10, 100]); ax_fdc.xaxis.set_major_formatter(plt.ScalarFormatter())
    lgrid(ax_fdc, alpha=0.08)
    plabel(ax_fdc, "e", x=-0.02, y=1.02)
    handles = ([Line2D([0],[0], color=SSP_COLORS[s], lw=1.4, label=SSP_LABELS[s]) for s in SCENARIOS]
               + [Line2D([0],[0], color="gray", ls="--", lw=0.8, label="Baseline"),
                  Line2D([0],[0], color="gray", lw=1.4, label="Controlled")])
    ax_fdc.legend(handles=handles, loc="upper right", fontsize=5, ncol=1, handlelength=1.2)

    for ext in ("png","pdf"): fig.savefig(FIG_DIR / f"fig2_timeseries_fdc.{ext}")
    plt.show()
else:
    print("  ⚠ Daily/FDC data not found — skipping Fig 2")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 3 — Surrogate Validation
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 3 — Surrogate validation")
print("=" * 60)

import glob as _glob

SURR_DIR = Path("outputs_surrogate_comparison")
C_WIN  = "#2166AC"; C_FULL = "#c72e29"
C_WIN_LIGHT = "#9ecae1"; C_FULL_LIGHT = "#fcae91"

_surr_files = sorted(_glob.glob(str(SURR_DIR / "surr_reach_*.parquet")))
if not _surr_files:
    print(f"  ⚠ No surr_reach_*.parquet in {SURR_DIR} — skipping Fig 3")
else:
    df_surr = pd.concat([pd.read_parquet(f) for f in _surr_files], ignore_index=True)
    df_surr_f = df_surr[
        (df_surr.epoch == "full") &
        (df_surr.fit_type.isin(["windowed_openloop", "full_openloop"]))
    ].copy()

    win_df = df_surr_f[df_surr_f.fit_type == "windowed_openloop"]
    ful_df = df_surr_f[df_surr_f.fit_type == "full_openloop"]

    reach_med = win_df.groupby("reach_id")["nse"].median().clip(-0.5, 1).reset_index()
    reach_med.columns = ["reach_id", "nse"]
    gdf_map3 = gdf_net.merge(reach_med, left_on="COMID",
                              right_on="reach_id", how="inner")
    print(f"  Reaches: {len(gdf_map3)}")

    fig3 = plt.figure(figsize=(W_FULL, 105*MM))
    gs_map3 = fig3.add_gridspec(1, 1, left=0.00, right=0.48,
                                bottom=0.00, top=0.95)
    gs_right3 = fig3.add_gridspec(2, 2, left=0.52, right=0.99,
                                  bottom=0.07, top=0.95,
                                  hspace=0.42, wspace=0.42,
                                  height_ratios=[1, 1])

    # ── (a) NSE map ──────────────────────────────────────────────
    ax_map3 = fig3.add_subplot(gs_map3[0, 0])
    ax_map3.set_aspect("equal"); ax_map3.axis("off")
    add_bg(ax_map3, gdf_net)

    cmap_nse3 = tcmap("RdYlGn", 0.05, 0.95)
    norm_nse3 = mcolors.TwoSlopeNorm(vcenter=0.5, vmin=-0.5, vmax=1.0)

    gdf_s3 = gdf_map3.sort_values("nse", ascending=True)
    lws3 = np.full(len(gdf_s3), 1)
    draw_net_cmap(ax_map3, gdf_s3, gdf_s3["nse"].values,
                  cmap_nse3, norm_nse3, lws3, alpha=0.88)

    b3 = gdf_net.total_bounds
    pad_x3 = (b3[2] - b3[0]) * 0.02; pad_y3 = (b3[3] - b3[1]) * 0.02
    ax_map3.set_xlim(b3[0] - pad_x3, b3[2] + pad_x3)
    ax_map3.set_ylim(b3[1] - pad_y3, b3[3] + pad_y3)
    add_scalebar(ax_map3, gdf_net, km=100)
    plabel_map(ax_map3, "a")

    cax3 = ax_map3.inset_axes([0.08, 0.03, 0.55, 0.03])
    sm3 = cm.ScalarMappable(cmap=cmap_nse3, norm=norm_nse3)
    cb3 = fig3.colorbar(sm3, cax=cax3, orientation="horizontal")
    cb3.set_label("NSE (ensemble median)", fontsize=6, labelpad=2)
    cb3.set_ticks([-0.5, 0, 0.5, 1.0])
    cb3.set_ticklabels(["-0.5", "0", "0.5", "1"])
    cb3.ax.tick_params(labelsize=5.5, width=0.3, length=2)
    cb3.outline.set_linewidth(0.3)

    # ── (b–c) CDFs ───────────────────────────────────────────────
    if "hydro" in win_df.columns:
        grp_keys3 = ["gcm", "hydro", "scenario"]
    else:
        grp_keys3 = ["gcm", "scenario"]

    cdf_configs3 = [
        ("nse", "NSE", (-0.25, 1.0), [0, 0.5], "b"),
        ("kge", "KGE", (-0.25, 1.0), [0, 0.5], "c"),
    ]

    for k, (metric, xlabel, xlim, vrefs, plab) in enumerate(cdf_configs3):
        ax = fig3.add_subplot(gs_right3[0, k])

        for _, sub in win_df.groupby(grp_keys3):
            vs = np.sort(sub[metric].dropna().values)
            if len(vs):
                ax.plot(vs, np.arange(1,len(vs)+1)/len(vs),
                        color=C_WIN_LIGHT, lw=0.25, alpha=0.35, zorder=0)
        for _, sub in ful_df.groupby(grp_keys3):
            vs = np.sort(sub[metric].dropna().values)
            if len(vs):
                ax.plot(vs, np.arange(1,len(vs)+1)/len(vs),
                        color=C_FULL_LIGHT, lw=0.25, alpha=0.30, zorder=0)

        vw = np.sort(win_df[metric].dropna().values)
        vf = np.sort(ful_df[metric].dropna().values)
        ax.plot(vw, np.arange(1,len(vw)+1)/len(vw),
                color=C_WIN, lw=1.4, label="Windowed", zorder=3)
        ax.plot(vf, np.arange(1,len(vf)+1)/len(vf),
                color=C_FULL, lw=1.4, ls="--", label="Full-period", zorder=3)

        for vr in vrefs:
            ax.axvline(vr, color="#ccc", ls=":", lw=0.5, zorder=0)

        mw = np.median(vw); mf = np.median(vf)
        ax.axvline(mw, color=C_WIN, ls=":", lw=0.7, alpha=0.6, zorder=2)
        ax.axvline(mf, color=C_FULL, ls=":", lw=0.7, alpha=0.6, zorder=2)

        ax.text(mw + 0.03, 0.92, f"{mw:.2f}",
                fontsize=5.5, ha="left", va="top", color=C_WIN,
                fontweight="bold", transform=ax.get_xaxis_transform(),
                bbox=dict(boxstyle="round,pad=0.1", fc="white",
                          ec="none", alpha=0.8), zorder=6)
        ax.text(mf + 0.03, 0.80, f"{mf:.2f}",
                fontsize=5.5, ha="left", va="top", color=C_FULL,
                fontweight="bold", transform=ax.get_xaxis_transform(),
                bbox=dict(boxstyle="round,pad=0.1", fc="white",
                          ec="none", alpha=0.8), zorder=6)

        ax.set_xlabel(xlabel)
        if k == 0: ax.set_ylabel("CDF (fraction of reaches)")
        ax.set_xlim(xlim); ax.set_ylim(0, 1)
        lgrid(ax, alpha=0.08)
        plabel(ax, plab, x=-0.12 if k == 0 else -0.08, y=1.12)
        if k == 0:
            ax.legend(fontsize=5.5, loc="upper left", handlelength=1.5)

    # ── (d) Multi-metric dot plot ────────────────────────────────
    ax_dot3 = fig3.add_subplot(gs_right3[1, :])

    metric_defs3 = [
        ("nse",   "NSE",      1),
        ("kge",   "KGE",      1),
        ("r2",    "R\u00b2",  1),
        ("pbias", "|PBIAS|", -1),
    ]
    metric_defs3 = [(c, l, d) for c, l, d in metric_defs3 if c in win_df.columns]

    yb3 = np.arange(len(metric_defs3))
    off3 = 0.17

    for i, (col, label, direction) in enumerate(metric_defs3):
        vw = win_df[col].dropna().values.copy()
        vf = ful_df[col].dropna().values.copy()
        if direction < 0:
            vw = np.abs(vw); vf = np.abs(vf)

        mw  = np.median(vw); q25w = np.percentile(vw, 25); q75w = np.percentile(vw, 75)
        mf  = np.median(vf); q25f = np.percentile(vf, 25); q75f = np.percentile(vf, 75)
        y_w = yb3[i] + off3; y_f = yb3[i] - off3

        ax_dot3.barh(y_w, q75w-q25w, left=q25w, height=0.30,
                     color=C_WIN, alpha=0.25, edgecolor="none")
        ax_dot3.plot(mw, y_w, "o", color=C_WIN, ms=4.5,
                     markeredgecolor="white", markeredgewidth=0.4, zorder=5)

        ax_dot3.barh(y_f, q75f-q25f, left=q25f, height=0.30,
                     color=C_FULL, alpha=0.25, edgecolor="none")
        ax_dot3.plot(mf, y_f, "s", color=C_FULL, ms=4,
                     markeredgecolor="white", markeredgewidth=0.4, zorder=5)

        x_label = max(q75w, q75f) + 0.02
        ax_dot3.text(x_label, y_w, f"{mw:.2f}", fontsize=5.5, va="center", color=C_WIN)
        ax_dot3.text(x_label, y_f, f"{mf:.2f}", fontsize=5.5, va="center", color=C_FULL)

    ax_dot3.set_yticks(yb3)
    ax_dot3.set_yticklabels([l for _, l, _ in metric_defs3])
    ax_dot3.set_xlabel("Metric value (median \u00b1 IQR)")
    ax_dot3.set_xlim(-0.05, 1.15)
    ax_dot3.invert_yaxis()
    lgrid(ax_dot3, alpha=0.08)
    plabel(ax_dot3, "d", x=-0.04, y=1.12)

    handles_dot3 = [
        Line2D([0],[0], marker="o", color="w", markerfacecolor=C_WIN,
               ms=4.5, markeredgecolor="white", markeredgewidth=0.3,
               label="Windowed (10-yr)"),
        Line2D([0],[0], marker="s", color="w", markerfacecolor=C_FULL,
               ms=4, markeredgecolor="white", markeredgewidth=0.3,
               label="Full-period"),
    ]
    ax_dot3.legend(handles=handles_dot3, loc="lower right", fontsize=5.5,
                   handletextpad=0.3)

    for ext in ("png","pdf"): fig3.savefig(FIG_DIR / f"fig3_validation.{ext}")
    plt.show()

    # ── Stats ────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  MANUSCRIPT STATS")
    print(f"{'='*65}")
    for m, lab in [("nse","NSE"), ("kge","KGE"), ("r2","R\u00b2"), ("pbias","PBIAS")]:
        if m not in win_df.columns: continue
        _mw = np.nanmedian(win_df[m]); _mf = np.nanmedian(ful_df[m])
        print(f"  {lab:>6}  windowed={_mw:.3f}  full={_mf:.3f}  \u0394={_mw-_mf:+.3f}")
    fw3 = np.nanmean(win_df["nse"] > 0.5)
    ff3 = np.nanmean(ful_df["nse"] > 0.5)
    print(f"  NSE>0.5  windowed={fw3:.1%}  full={ff3:.1%}  \u0394={fw3-ff3:+.1%}")
    print(f"{'='*65}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 4 — Aggregate Performance
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 4 — Aggregate performance")
print("=" * 60)

if has_ens:
    fig = plt.figure(figsize=(W_FULL, 80*MM))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.2, 1],
                  hspace=0.35, wspace=0.35,
                  left=0.08, right=0.97, bottom=0.10, top=0.96)
    ax_a = fig.add_subplot(gs[:, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 1])
else:
    fig, ax_a = plt.subplots(figsize=(W_HALF, 70*MM))

# (a) Pareto
hydro_col = ["hydro"] if "hydro" in df_all.columns else []
basin = (df_all.groupby(["scenario","gcm"] + hydro_col + ["R_weight"], as_index=False)
         .agg(eff=("eff_total_L1","sum"), res=("vol_hi_controlled","sum")))
basin["eff_gm3"] = basin["eff"]/1e9; basin["res_gm3"] = basin["res"]/1e9
summ = (basin.groupby(["scenario","R_weight"], as_index=False)
        .agg(em=("eff_gm3","median"), rm=("res_gm3","median"),
             rq25=("res_gm3", lambda x: x.quantile(0.25)),
             rq75=("res_gm3", lambda x: x.quantile(0.75))))

for scn in SCENARIOS:
    s = summ[summ["scenario"]==scn].sort_values("em")
    ax_a.fill_between(s["em"], s["rq25"], s["rq75"], alpha=0.15, color=SSP_COLORS[scn])
    ax_a.plot(s["em"], s["rm"], color=SSP_COLORS[scn], lw=1.4, label=SSP_LABELS[scn])
ax_a.axvline(TARGET_BUDGET, color="#777", lw=0.8, ls="--")
ax_a.text(TARGET_BUDGET+0.3, ax_a.get_ylim()[1]*0.92,
          f"Budget = {TARGET_BUDGET:g}", color="#666", fontsize=6, va="top")
ax_a.set_xlabel(r"Effort (Gm$^3$ /yr)")
ax_a.set_ylabel(r"Basin residual flood volume (Gm$^3$ /yr)")
ax_a.legend(loc="upper right", fontsize=5.5, ncol=2)
lgrid(ax_a); plabel(ax_a, "a", x=-0.10, y=1.03)

if has_ens:
    # (b) Multi-metric dot
    d_gcm = ens_gcm[ens_gcm["budget_x"]==TARGET_BUDGET].copy()
    ref_g = d_gcm[d_gcm["scenario"]==REF]
    metrics = [(c,l) for c,l in [
        ("vol_controlled_Gm3yr","Volume"),("peak_controlled_sum","Annual Peak"),
        ("rl10_controlled_sum","RL10"),("rl100_controlled_sum","RL100")]
        if c in d_gcm.columns]
    ref_v = {c: ref_g[c].median() for c,_ in metrics}
    yb = np.arange(len(metrics)); off = 0.20
    for i,scn in enumerate(SCENARIOS):
        gd = d_gcm[d_gcm["scenario"]==scn]
        for j,(col,_) in enumerate(metrics):
            if ref_v[col]==0: continue
            g = gd[col]/ref_v[col]; md=g.median(); q25=g.quantile(0.25); q75=g.quantile(0.75)
            y = yb[j]+(i-1.5)*off
            ax_b.barh(y, q75-q25, left=q25, height=0.14,
                      color=SSP_COLORS[scn], alpha=0.40, edgecolor="none")
            ax_b.plot(md, y, "o", color=SSP_COLORS[scn], ms=4.5,
                      markeredgecolor="white", markeredgewidth=0.3, zorder=5)
    ax_b.axvline(1.0, color="#aaa", ls="--", lw=0.6, zorder=0)
    ax_b.set_yticks(yb); ax_b.set_yticklabels([l for _,l in metrics])
    ax_b.set_xlabel("Residual risk ratio (vs SSP1-2.6)")
    lgrid(ax_b, alpha=0.08); plabel(ax_b, "b", x=-0.2, y=1.03)

    # (c) RL spectrum
    d = ens_summ[ens_summ["budget_x"]==TARGET_BUDGET].copy()
    rps = [2,5,10,20,50,100]
    for scn in SCENARIOS:
        row = d[d["scenario"]==scn]
        if len(row)==0: continue
        row = row.iloc[0]
        ym  = [row.get(f"rl{T}_controlled_sum_gcm_med", np.nan)/1e6 for T in rps]
        y25 = [row.get(f"rl{T}_controlled_sum_gcm_q25", np.nan)/1e6 for T in rps]
        y75 = [row.get(f"rl{T}_controlled_sum_gcm_q75", np.nan)/1e6 for T in rps]
        ax_c.fill_between(rps, y25, y75, alpha=0.15, color=SSP_COLORS[scn])
        ax_c.plot(rps, ym, "o-", color=SSP_COLORS[scn], lw=1.0, ms=3, label=SSP_LABELS[scn])
    ax_c.set_xscale("log"); ax_c.set_xticks(rps)
    ax_c.xaxis.set_major_formatter(plt.ScalarFormatter())
    ax_c.set_xlabel("Return period (years)")
    ax_c.set_ylabel(r"Basin-total return level ($\times 10^6$ m$^3$/s)")
    ax_c.legend(fontsize=5, loc="upper left")
    lgrid(ax_c, alpha=0.08); plabel(ax_c, "c", x=-0.2, y=1.06)

for ext in ("png","pdf"): fig.savefig(FIG_DIR / f"fig4_aggregate.{ext}")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 5 — Residual Ratio Maps (flow-weighted)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 5 — Residual ratio maps")
print("=" * 60)

FIG5_REF  = REF
FIG5_COMP = [245, 370, 585]

df_ref5 = df_ens[df_ens["scenario"]==FIG5_REF][["reach_id","vol_hi_controlled","vol_hi_baseline"]].copy()
df_ref5.columns = ["reach_id","ctrl_ref","bl_ref"]

gdf_rat = {}
for scn in FIG5_COMP:
    ds = df_ens[df_ens["scenario"]==scn][["reach_id","vol_hi_controlled","vol_hi_baseline"]].copy()
    ds.columns = ["reach_id", f"ctrl_{scn}", f"bl_{scn}"]
    m = df_ref5.merge(ds, on="reach_id")
    m["res_ratio"] = m[f"ctrl_{scn}"] / (m["ctrl_ref"] + 1e-6)
    m["flow_weight"] = (m["bl_ref"] + m[f"bl_{scn}"]) / 2
    gdf_rat[scn] = gdf_net.merge(m, left_on="COMID", right_on="reach_id", how="inner")

fig = plt.figure(figsize=(W_FULL, 85*MM))
gs = GridSpec(2, 3, figure=fig, height_ratios=[3.5, 1.5],
              hspace=0.22, wspace=0.03,
              left=0.01, right=0.89, bottom=0.08, top=0.94)

RATIO_LO, RATIO_HI = -2, 2
RATIO_TICKS = [-2, -1, 0, 1, 2]
RATIO_TLABS = [r"$\div$4", r"$\div$2", "1:1", r"$\times$2", r"$\times$4"]
cmap5 = tcmap("RdBu_r", 0.05, 0.95)
norm5 = mcolors.TwoSlopeNorm(vcenter=0, vmin=RATIO_LO, vmax=RATIO_HI)

for k, (scn, lab) in enumerate(zip(FIG5_COMP, ["a","b","c"])):
    ax = fig.add_subplot(gs[0, k])
    ax.set_aspect("equal"); ax.axis("off")
    add_bg(ax, gdf_net)
    gdf_p = gdf_rat[scn].sort_values("TotDASqKM", ascending=True)
    lw = _flow_weighted_lw(gdf_p)
    log2r = np.log2(np.clip(gdf_p["res_ratio"].to_numpy(float), 2**RATIO_LO, 2**RATIO_HI))
    draw_net_cmap(ax, gdf_p, log2r, cmap5, norm5, lw)
    plabel_map(ax, lab)
    ax.text(0.50, 0.01, SSP_LABELS[scn], transform=ax.transAxes, ha="center",
            fontsize=5, fontweight="bold", color=SSP_COLORS[scn],
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))
    if k == 0: add_scalebar(ax, gdf_net)

make_colorbar(fig, [0.91, 0.40, 0.012, 0.50], cmap5, norm5,
              r"Residual ratio vs SSP1-2.6 ($\log_2$)",
              ticks=RATIO_TICKS, ticklabels=RATIO_TLABS)

# (d) Stream-order summary
ax_d = fig.add_subplot(gs[1, :])
so_groups = [("Order 1–2",[1,2]), ("Order 3–4",[3,4]),
             ("Order 5+",[5,6,7,8,9]), ("All",None)]
yb = np.arange(len(so_groups)); off = 0.22
for i, scn in enumerate(FIG5_COMP):
    g = gdf_rat[scn].copy(); g["so"] = g["StreamOrde"].astype(int)
    for j, (glabel, orders) in enumerate(so_groups):
        sub = g[g["so"].isin(orders)] if orders else g
        lr = np.log2(np.clip(sub["res_ratio"].to_numpy(float), 0.1, 50))
        fw = sub["flow_weight"].to_numpy(float)
        mask = np.isfinite(lr) & np.isfinite(fw) & (fw > 0)
        lr, fw = lr[mask], fw[mask]
        if len(lr) < 3: continue
        md  = _weighted_quantile(lr, fw, 0.50)
        q25 = _weighted_quantile(lr, fw, 0.25)
        q75 = _weighted_quantile(lr, fw, 0.75)
        y = yb[j] + (i-1)*off
        ax_d.barh(y, q75-q25, left=q25, height=0.16,
                  color=SSP_COLORS[scn], alpha=0.40, edgecolor="none")
        ax_d.plot(md, y, "o", color=SSP_COLORS[scn], ms=4.5,
                  markeredgecolor="white", markeredgewidth=0.3, zorder=5)

ax_d.axvline(0, color="#aaa", ls="--", lw=0.6, zorder=0)
ax_d.set_yticks(yb); ax_d.set_yticklabels([gl for gl,_ in so_groups])
ax_d.set_xlabel(r"Residual ratio vs SSP1-2.6 ($\log_2$)")
ax_d.set_xticks([0,1]); ax_d.set_xticklabels(["1:1", r"$\times$2"])
ax_d.set_xlim(-0.5, 1.5)
lgrid(ax_d, alpha=0.08); plabel(ax_d, "d", x=-0.03, y=1.10)
handles_d = [Patch(facecolor=SSP_COLORS[s], alpha=0.5, label=SSP_LABELS[s]) for s in FIG5_COMP]
ax_d.legend(handles=handles_d, loc="lower right", fontsize=5, ncol=3)

for ext in ("png","pdf"): fig.savefig(FIG_DIR / f"fig5_ratio_maps.{ext}")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 6 — EFFORT ALLOCATION STRUCTURE  (4-panel, Q̄-unified)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 6 — Effort allocation structure")
print("=" * 60)

SCN_FIG6 = 245
sub6 = df_ens[df_ens["scenario"] == SCN_FIG6].copy()

# ── Effort intensity (effort / mean flow) + residual ─────────────
eff_raw6 = sub6["eff_total_L1"].to_numpy(float)
da6      = sub6["TotDASqKM"].to_numpy(float)
mf6      = sub6["mean_flow_ref"].to_numpy(float)

sub6["eff_intensity"] = eff_raw6 / mf6

m6 = (eff_raw6 > 0) & (mf6 > 0) & np.isfinite(eff_raw6) & np.isfinite(mf6)
log_eff6, log_mf6 = np.log10(eff_raw6[m6]), np.log10(mf6[m6])
lr6 = linregress(log_mf6, log_eff6)
resid6 = log_eff6 - (lr6.slope * log_mf6 + lr6.intercept)

sub6_m = sub6.iloc[np.where(m6)[0]].copy()
sub6_m["log_eff_resid"] = resid6

# ── R² stats ─────────────────────────────────────────────────────
r2_eff_mf = lr6.rvalue**2
m_da = m6 & (da6 > 0) & np.isfinite(da6)
lr6_da = linregress(np.log10(da6[m_da]), np.log10(eff_raw6[m_da]))
r2_eff_da = lr6_da.rvalue**2
print(f"  R²(log effort ~ log Q̄)={r2_eff_mf:.2f}  R²(log effort ~ log DA)={r2_eff_da:.2f}")

# ── Top-10% sets ─────────────────────────────────────────────────
pct_thresh = 10
n6 = len(sub6)
k = max(1, int(n6 * pct_thresh / 100))

rank_lqr = np.argsort(-eff_raw6)
rank_mf  = np.argsort(-mf6)
rank_da  = np.argsort(-da6)

top_lqr_set = set(rank_lqr[:k])
top_mf_set  = set(rank_mf[:k])
top_da_set  = set(rank_da[:k])

# ── Consistent 5-class: LQR unique / Q̄ unique / DA unique / overlap / none
mismatch_cat = np.zeros(n6, dtype=int)
for i in range(n6):
    in_lqr = i in top_lqr_set
    in_mf  = i in top_mf_set
    in_da  = i in top_da_set
    count  = in_lqr + in_mf + in_da
    if count >= 2:
        mismatch_cat[i] = 4      # overlap
    elif in_lqr:
        mismatch_cat[i] = 1      # LQR unique
    elif in_mf:
        mismatch_cat[i] = 2      # Q̄ unique
    elif in_da:
        mismatch_cat[i] = 3      # DA unique

sub6["mismatch_cat"] = mismatch_cat
n_lqr_uniq = (mismatch_cat == 1).sum()
n_mf_uniq  = (mismatch_cat == 2).sum()
n_da_uniq  = (mismatch_cat == 3).sum()
n_overlap  = (mismatch_cat == 4).sum()

# ── Residual / baseline by unique category ───────────────────────
resid_vol = sub6["vol_hi_controlled"].fillna(0).values
bl_vol    = sub6["vol_hi_baseline"].fillna(0).values

def resid_ratio(idx):
    if len(idx) == 0: return 0
    return resid_vol[idx].mean() / max(bl_vol[idx].mean(), 1)

idx_lqr = np.where(mismatch_cat == 1)[0]
idx_mf  = np.where(mismatch_cat == 2)[0]
idx_da  = np.where(mismatch_cat == 3)[0]

rr_lqr = resid_ratio(idx_lqr)
rr_mf  = resid_ratio(idx_mf)
rr_da  = resid_ratio(idx_da)

print(f"  Unique counts: LQR={n_lqr_uniq}, Q̄={n_mf_uniq}, DA={n_da_uniq}, overlap={n_overlap}")
print(f"  Resid/baseline — LQR: {rr_lqr:.2f}, Q̄: {rr_mf:.2f}, DA: {rr_da:.2f}")

# ── Colors (shared between map and bar) ──────────────────────────
cat_colors = {
    0: "#e8e8e8",   # none
    1: "#d62728",   # LQR unique — red
    2: "#1f77b4",   # Q̄ unique — blue
    3: "#2ca02c",   # DA unique — green
    4: "#bbb",      # overlap — gray
}

# ── Shared map extent ────────────────────────────────────────────
_bnds = gdf_net.total_bounds
_xpad = (_bnds[2] - _bnds[0]) * 0.02
_ypad = (_bnds[3] - _bnds[1]) * 0.02
_map_xlim = (_bnds[0] - _xpad, _bnds[2] + _xpad)
_map_ylim = (_bnds[1] - _ypad, _bnds[3] + _ypad)
_UNI_LW = 0.95

# ── DRAW: 2×2 layout ────────────────────────────────────────────
fig6 = plt.figure(figsize=(W_FULL, 160*MM))
gs6 = GridSpec(2, 2, figure=fig6, height_ratios=[1, 1],
               hspace=0.08, wspace=0.18,
               left=0.03, right=0.97, bottom=0.07, top=0.97)

# ── (a) Raw effort map ──────────────────────────────────────────
ax6a = fig6.add_subplot(gs6[0, 0])
ax6a.set_aspect("equal"); ax6a.axis("off"); add_bg(ax6a, gdf_net)

gdf_eff6 = gdf_net.merge(sub6[["reach_id","eff_total_L1"]],
                          left_on="COMID", right_on="reach_id", how="inner")
gdf_eff6 = gdf_eff6.sort_values("TotDASqKM", ascending=True)
gdf_eff6["log_eff"] = np.log10(gdf_eff6["eff_total_L1"].clip(lower=1))

cmap_eff6 = tcmap("YlOrRd", 0.10, 0.95)
vmin_eff6 = np.nanpercentile(gdf_eff6["log_eff"].values, 5)
vmax_eff6 = np.nanpercentile(gdf_eff6["log_eff"].values, 95)
norm_eff6 = mcolors.Normalize(vmin=vmin_eff6, vmax=vmax_eff6)
lw6a = np.full(len(gdf_eff6), _UNI_LW)
draw_net_cmap(ax6a, gdf_eff6, gdf_eff6["log_eff"].values,
              cmap_eff6, norm_eff6, lw6a)
ax6a.set_xlim(_map_xlim); ax6a.set_ylim(_map_ylim)
plabel_map(ax6a, "a"); add_scalebar(ax6a, gdf_net)

cax6a = ax6a.inset_axes([0.05, 0.04, 0.38, 0.025])
cb6a = fig6.colorbar(cm.ScalarMappable(cmap=cmap_eff6, norm=norm_eff6),
                     cax=cax6a, orientation="horizontal")
cb6a.set_label(r"Raw effort (log$_{10}$ m$^3$ yr$^{-1}$)", fontsize=7, labelpad=2)
cb6a.ax.tick_params(labelsize=6.5, width=0.3, length=1.5); cb6a.outline.set_linewidth(0.3)

# ── (b) Effort intensity: effort / Q̄ ────────────────────────────
ax6b = fig6.add_subplot(gs6[0, 1])
ax6b.set_aspect("equal"); ax6b.axis("off"); add_bg(ax6b, gdf_net)

gdf_ni6 = gdf_net.merge(sub6[["reach_id", "eff_intensity"]],
                         left_on="COMID", right_on="reach_id", how="inner")
gdf_ni6 = gdf_ni6.sort_values("TotDASqKM", ascending=True)
gdf_ni6["log_intensity"] = np.log10(gdf_ni6["eff_intensity"].clip(lower=1e-12))

cmap_ni6 = tcmap("magma_r", 0.05, 0.92)
vmin_ni6 = np.nanpercentile(gdf_ni6["log_intensity"].values, 5)
vmax_ni6 = np.nanpercentile(gdf_ni6["log_intensity"].values, 95)
norm_ni6 = mcolors.Normalize(vmin=vmin_ni6, vmax=vmax_ni6)
lw6b = np.full(len(gdf_ni6), _UNI_LW)
draw_net_cmap(ax6b, gdf_ni6, gdf_ni6["log_intensity"].values,
              cmap_ni6, norm_ni6, lw6b)
ax6b.set_xlim(_map_xlim); ax6b.set_ylim(_map_ylim)
plabel_map(ax6b, "b")

cax6b = ax6b.inset_axes([0.55, 0.02, 0.38, 0.025])
cb6b = fig6.colorbar(cm.ScalarMappable(cmap=cmap_ni6, norm=norm_ni6),
                     cax=cax6b, orientation="horizontal")
cb6b.set_label(r"Effort / $\overline{Q}$ (log$_{10}$, dimensionless)",
               fontsize=7, labelpad=2)
cb6b.ax.tick_params(labelsize=6.5, width=0.3, length=1.5); cb6b.outline.set_linewidth(0.3)

# ── (c) Q̄-detrended residual ────────────────────────────────────
ax6c = fig6.add_subplot(gs6[1, 0])
ax6c.set_aspect("equal"); ax6c.axis("off")
gdf_net.plot(ax=ax6c, color="#e8e8e8", linewidth=0.10, alpha=0.45, zorder=0)

gdf_r6 = gdf_net.merge(sub6_m[["reach_id","log_eff_resid"]],
                        left_on="COMID", right_on="reach_id", how="inner")
gdf_r6 = gdf_r6.sort_values("TotDASqKM", ascending=True)

cmap_r6 = tcmap("RdBu_r", 0.05, 0.95)
rlim6 = np.nanpercentile(np.abs(gdf_r6["log_eff_resid"].values), 95)
norm_r6 = mcolors.TwoSlopeNorm(vcenter=0, vmin=-rlim6, vmax=rlim6)
lw6c = np.full(len(gdf_r6), _UNI_LW)
draw_net_cmap(ax6c, gdf_r6, gdf_r6["log_eff_resid"].values,
              cmap_r6, norm_r6, lw6c)
ax6c.set_xlim(_map_xlim); ax6c.set_ylim(_map_ylim)
plabel_map(ax6c, "c")

cax6c = ax6c.inset_axes([0.05, 0.04, 0.38, 0.025])
cb6c = fig6.colorbar(cm.ScalarMappable(cmap=cmap_r6, norm=norm_r6),
                     cax=cax6c, orientation="horizontal")
cb6c.set_label(r"Effort residual from $\overline{Q}$ scaling (log$_{10}$)",
               fontsize=7, labelpad=2)
cb6c.ax.tick_params(labelsize=6.5, width=0.3, length=1.5); cb6c.outline.set_linewidth(0.3)

# ── (d) Mismatch map + legend ────────────────────────────────────
ax6d = fig6.add_subplot(gs6[1, 1])
ax6d.set_aspect("equal"); ax6d.axis("off")

gdf_mm = gdf_net.merge(sub6[["reach_id", "mismatch_cat"]],
                        left_on="COMID", right_on="reach_id", how="inner")

# draw order: background → overlap → DA unique → Q̄ unique → LQR unique
gdf_c0 = gdf_mm[gdf_mm["mismatch_cat"] == 0]
if len(gdf_c0) > 0:
    gdf_c0.plot(ax=ax6d, color=cat_colors[0], linewidth=0.05, alpha=0.20, zorder=0)

gdf_c4 = gdf_mm[gdf_mm["mismatch_cat"] == 4]
if len(gdf_c4) > 0:
    gdf_c4.plot(ax=ax6d, color=cat_colors[4], linewidth=1.0, alpha=0.50, zorder=1)

gdf_c3 = gdf_mm[gdf_mm["mismatch_cat"] == 3]
if len(gdf_c3) > 0:
    gdf_c3.plot(ax=ax6d, color="white", linewidth=3.0, alpha=1.0, zorder=2)
    gdf_c3.plot(ax=ax6d, color=cat_colors[3], linewidth=1.8, alpha=0.85, zorder=3)

gdf_c2 = gdf_mm[gdf_mm["mismatch_cat"] == 2]
if len(gdf_c2) > 0:
    gdf_c2.plot(ax=ax6d, color="white", linewidth=3.2, alpha=1.0, zorder=4)
    gdf_c2.plot(ax=ax6d, color=cat_colors[2], linewidth=2.0, alpha=0.88, zorder=5)

gdf_c1 = gdf_mm[gdf_mm["mismatch_cat"] == 1]
if len(gdf_c1) > 0:
    gdf_c1.plot(ax=ax6d, color="white", linewidth=4.2, alpha=1.0, zorder=6)
    gdf_c1.plot(ax=ax6d, color=cat_colors[1], linewidth=2.8, alpha=0.95, zorder=7)

ax6d.set_xlim(_map_xlim); ax6d.set_ylim(_map_ylim)
plabel_map(ax6d, "d")

legend_handles = [
    Line2D([0],[0], color=cat_colors[1], lw=3.0,
           label=f"LQR unique ({n_lqr_uniq})"),
    Line2D([0],[0], color=cat_colors[2], lw=2.0,
           label=f"$\\overline{{Q}}$ unique ({n_mf_uniq})"),
    Line2D([0],[0], color=cat_colors[3], lw=1.8,
           label=f"DA unique ({n_da_uniq})"),
    Line2D([0],[0], color=cat_colors[4], lw=1.0, alpha=0.50,
           label=f"Overlap ({n_overlap})"),
]
ax6d.legend(handles=legend_handles, loc="lower left", fontsize=6,
            frameon=False, handlelength=2.0, labelspacing=0.25, borderpad=0.3)

for ext in ("png","pdf"): fig6.savefig(FIG_DIR / f"fig6_effort_structure.{ext}")
plt.show()
print(f"\n  R²(Q̄)={r2_eff_mf:.3f}  R²(DA)={r2_eff_da:.3f}")
print(f"  Resid/baseline — LQR: {rr_lqr:.2f}, Q̄: {rr_mf:.2f}, DA: {rr_da:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 7 — Main-Stem Profiles
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 7 — Main-stem profiles")
print("=" * 60)

lpi = gdf_net.loc[gdf_net["TotDASqKM"].idxmax(), "LevelPathI"]
ms = gdf_net[gdf_net["LevelPathI"] == lpi].copy()
if "Divergence" in ms.columns: ms = ms[ms["Divergence"] != 2]
if "Hydroseq"  in ms.columns: ms = ms.sort_values("Hydroseq", ascending=False)
ms_ids = set(ms["COMID"].astype(int))
has_rl10 = "rl10_controlled" in df_sel.columns

vol_median = (df_sel[df_sel["reach_id"].isin(ms_ids)]
              .groupby("reach_id")["vol_hi_controlled"].median())
ghost_ids = set(vol_median[vol_median < 1e6].index)
ms_ids -= ghost_ids

ms_clean = (ms[ms["COMID"].astype(int).isin(ms_ids)]
            [["COMID","TotDASqKM","Hydroseq"]].copy())
ms_clean["COMID"] = ms_clean["COMID"].astype(int)
ms_clean = ms_clean.sort_values("Hydroseq", ascending=False).reset_index(drop=True)
ms_clean = ms_clean.drop_duplicates(subset="TotDASqKM", keep="first")
ms_lookup = ms_clean.set_index("COMID")["TotDASqKM"]
valid_ms_ids = set(ms_lookup.index)
print(f"  Main-stem reaches: {len(valid_ms_ids)}")

rows_vol = []; rows_rl10 = []
member_iter = ([(gcm, hydro)
                for gcm in df_sel["gcm"].unique()
                for hydro in df_sel["hydro"].unique()]
               if "hydro" in df_sel.columns
               else [(gcm, None) for gcm in GCMS])

for gcm, hydro in member_iter:
    for scn in SCENARIOS:
        mask = (df_sel["gcm"]==gcm) & (df_sel["scenario"]==scn)
        if hydro is not None: mask = mask & (df_sel["hydro"]==hydro)
        sub = df_sel[mask & df_sel["reach_id"].isin(valid_ms_ids)].copy()
        if len(sub) < 5: continue
        sub["da"] = sub["reach_id"].map(ms_lookup)
        sub = sub.dropna(subset=["da"]).sort_values("da")
        for _, r in sub.iterrows():
            rows_vol.append({"da": r["da"], "scn": scn,
                             "val": r["vol_hi_controlled"]/1e6})
            if has_rl10:
                rows_rl10.append({"da": r["da"], "scn": scn,
                                  "val": r["rl10_controlled"]/1e3})

df_vol7 = pd.DataFrame(rows_vol)
df_rl10_7 = pd.DataFrame(rows_rl10) if rows_rl10 else None

def _reach_stats(df):
    return (df.groupby(["scn","da"])["val"]
            .agg(med="median",
                 q25=lambda x: x.quantile(0.25),
                 q75=lambda x: x.quantile(0.75))
            .reset_index().sort_values("da"))

sv7 = _reach_stats(df_vol7)
sr7 = _reach_stats(df_rl10_7) if df_rl10_7 is not None else None

fig7, (ax7a, ax7b) = plt.subplots(2, 1, figsize=(W_FULL, 95*MM), sharex=True,
                                   gridspec_kw={"hspace": 0.15,
                                                "height_ratios": [1, 1]})
for scn in SCENARIOS:
    d = sv7[sv7["scn"]==scn]
    ax7a.fill_between(d["da"], d["q25"], d["q75"], alpha=0.12, color=SSP_COLORS[scn])
    ax7a.plot(d["da"], d["med"], "-", color=SSP_COLORS[scn], lw=1.0, alpha=0.6)
    ax7a.scatter(d["da"], d["med"], s=3, color=SSP_COLORS[scn],
                 edgecolors="white", linewidths=0.3, zorder=5, label=SSP_LABELS[scn])
ax7a.set_ylabel(r"Residual volume ($\times\,10^6$ m$^3$/yr)")
ax7a.legend(loc="upper left", ncol=4, fontsize=5.5)
ax7a.yaxis.set_major_locator(ticker.MaxNLocator(5))
lgrid(ax7a); plabel(ax7a, "a", x=-0.07, y=1.08)
plt.setp(ax7a.get_xticklabels(), visible=False)

if sr7 is not None and len(sr7) > 0:
    for scn in SCENARIOS:
        d = sr7[sr7["scn"]==scn]
        ax7b.fill_between(d["da"], d["q25"], d["q75"], alpha=0.12, color=SSP_COLORS[scn])
        ax7b.plot(d["da"], d["med"], "-", color=SSP_COLORS[scn], lw=1.0, alpha=0.6)
        ax7b.scatter(d["da"], d["med"], s=3, color=SSP_COLORS[scn],
                     edgecolors="white", linewidths=0.3, zorder=5)
    ax7b.set_ylabel(r"RL10 residual ($\times\,10^3$ m$^3$/s)")
else:
    dref7 = sv7[sv7["scn"]==REF].set_index("da")["med"]
    for scn in [245,370,585]:
        d = sv7[sv7["scn"]==scn].copy()
        d["ratio"] = d.set_index("da")["med"] / dref7.reindex(d["da"].values).values
        ax7b.plot(d["da"], d["ratio"], "-", color=SSP_COLORS[scn], lw=1.0, alpha=0.6)
        ax7b.scatter(d["da"], d["ratio"], s=10, color=SSP_COLORS[scn],
                     edgecolors="white", linewidths=0.3, zorder=5)
    ax7b.axhline(1.0, color="#999", ls=":", lw=0.5)
    ax7b.set_ylabel("Residual ratio\n(vs SSP1-2.6)")

ax7b.set_xlabel(r"Drainage area (km$^2$)")
ax7b.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}k" if x>=1e3 else f"{x:.0f}"))
lgrid(ax7b); plabel(ax7b, "b", x=-0.07, y=1.08)
fig7.align_ylabels()

for ext in ("png","pdf"): fig7.savefig(FIG_DIR / f"fig7_mainstem.{ext}")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIG 8 — Control Effectiveness + GCM Agreement
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG 8 — Control effectiveness — ensemble agreement")
print("=" * 60)

FIG8_REF  = 245
FIG8_COMP = 585

def _fig8_ens_median(df_b, metric_col):
    cols = []
    if "hydro" in df_b.columns:
        for (gcm, hydro), sub in df_b.groupby(["gcm","hydro"]):
            cols.append(sub.set_index("reach_id")[metric_col].rename(f"{gcm}_{hydro}"))
    else:
        for gcm in df_b["gcm"].unique():
            cols.append(df_b[df_b["gcm"]==gcm].set_index("reach_id")[metric_col].rename(gcm))
    return pd.concat(cols, axis=1).median(axis=1) if cols else pd.Series(dtype=float)

def _fig8_per_member(df_b, metric_col):
    cols = []
    if "hydro" in df_b.columns:
        for (gcm, hydro), sub in df_b.groupby(["gcm","hydro"]):
            cols.append(sub.set_index("reach_id")[metric_col].rename(f"{gcm}_{hydro}"))
    else:
        for gcm in df_b["gcm"].unique():
            cols.append(df_b[df_b["gcm"]==gcm].set_index("reach_id")[metric_col].rename(gcm))
    return pd.concat(cols, axis=1) if cols else pd.DataFrame()

if "peak_red_rel" not in df_sel.columns:
    print("  ⚠ peak_red_rel missing — skipping Fig 8")
else:
    df8_ref = df_sel[df_sel["scenario"]==FIG8_REF]
    df8_585 = df_sel[df_sel["scenario"]==FIG8_COMP]

    red_ref8 = _fig8_ens_median(df8_ref, "peak_red_rel")
    red_585_8 = _fig8_ens_median(df8_585, "peak_red_rel")

    wide_ref8 = _fig8_per_member(df8_ref, "peak_red_rel")
    wide_585_8 = _fig8_per_member(df8_585, "peak_red_rel")
    common_m8 = list(set(wide_ref8.columns) & set(wide_585_8.columns))
    n_mem8 = len(common_m8)
    print(f"  Ensemble members: {n_mem8}")

    delta_wide8 = wide_585_8[common_m8] - wide_ref8[common_m8]
    n_degrade8 = (delta_wide8 < 0).sum(axis=1)

    df8 = pd.DataFrame({
        "red_ref": red_ref8, "red_585": red_585_8, "n_degrade": n_degrade8,
    }).dropna().reset_index().rename(columns={"index": "reach_id"})
    df8["red_ref"]  = df8["red_ref"].clip(-0.2, 1.0)
    df8["red_585"]  = df8["red_585"].clip(-0.2, 1.0)
    df8["delta_red"] = df8["red_585"] - df8["red_ref"]

    pct_degrade = (df8["delta_red"] < 0).mean() * 100
    print(f"  Reaches: {len(df8):,}  Δ median={df8['delta_red'].median()*100:.1f}pp")
    print(f"  {pct_degrade:.1f}% of reaches show degradation under SSP5-8.5")

    AGREE_CMAP8, AGREE_NORM8 = make_agreement_cmap(n_mem8)
    gdf8 = gdf_net.merge(df8, left_on="COMID", right_on="reach_id", how="inner")

    _bnds8 = gdf_net.total_bounds
    _xp8 = (_bnds8[2]-_bnds8[0])*0.02; _yp8 = (_bnds8[3]-_bnds8[1])*0.02
    _mx8 = (_bnds8[0]-_xp8, _bnds8[2]+_xp8)
    _my8 = (_bnds8[1]-_yp8, _bnds8[3]+_yp8)

    # ── FIGURE ───────────────────────────────────────────────────
    fig8 = plt.figure(figsize=(W_FULL, 90*MM))

    gs_maps = fig8.add_gridspec(1, 2, left=0.02, right=0.68,
                                 bottom=0.10, top=0.95, wspace=0.06)

    # ══ (a) Δ peak reduction map ════════════════════════════════
    ax8a = fig8.add_subplot(gs_maps[0, 0])
    ax8a.set_aspect("equal"); ax8a.axis("off"); add_bg(ax8a, gdf_net)
    cmap_div8 = tcmap("RdBu", 0.05, 0.95)
    dr8 = gdf8["delta_red"].values
    dr_lim8 = max(np.nanpercentile(np.abs(dr8[np.isfinite(dr8)]), 95), 0.05)
    norm_dr8 = mcolors.TwoSlopeNorm(vcenter=0, vmin=-dr_lim8, vmax=dr_lim8)
    gdf_s8a = gdf8.iloc[np.argsort(np.abs(gdf8["delta_red"].values))]
    draw_net_cmap(ax8a, gdf_s8a, gdf_s8a["delta_red"].values,
                  cmap_div8, norm_dr8, lw_order(gdf_s8a), alpha=0.85)
    ax8a.set_xlim(_mx8); ax8a.set_ylim(_my8)
    add_scalebar(ax8a, gdf_net, km=100); plabel_map(ax8a, "a")

    cax8a = ax8a.inset_axes([0.10, 0.02, 0.45, 0.030])
    cb8a = fig8.colorbar(cm.ScalarMappable(cmap=cmap_div8, norm=norm_dr8),
                         cax=cax8a, orientation="horizontal")
    cb8a.set_label("Δ peak reduction (pp)", fontsize=7, labelpad=2)
    _dr_t8 = [-dr_lim8, -dr_lim8/2, 0, dr_lim8/2, dr_lim8]
    cb8a.set_ticks(_dr_t8); cb8a.set_ticklabels([f"{t*100:+.0f}" for t in _dr_t8])
    cb8a.ax.tick_params(labelsize=6.5, width=0.3, length=1.5); cb8a.outline.set_linewidth(0.3)

    # ══ (b) Ensemble agreement map ══════════════════════════════
    ax8b = fig8.add_subplot(gs_maps[0, 1])
    ax8b.set_aspect("equal"); ax8b.axis("off"); add_bg(ax8b, gdf_net)
    gdf8["agree_ext"] = np.abs(gdf8["n_degrade"] - n_mem8/2)
    gdf_s8b = gdf8.sort_values("agree_ext", ascending=True)
    draw_net_cmap(ax8b, gdf_s8b, gdf_s8b["n_degrade"].values,
                  AGREE_CMAP8, AGREE_NORM8, lw_order(gdf_s8b), alpha=0.88)
    ax8b.set_xlim(_mx8); ax8b.set_ylim(_my8)
    plabel_map(ax8b, "b")

    cax8b = ax8b.inset_axes([0.10, 0.02, 0.45, 0.030])
    cb8b = fig8.colorbar(cm.ScalarMappable(cmap=AGREE_CMAP8, norm=AGREE_NORM8),
                         cax=cax8b, orientation="horizontal", spacing="uniform")
    ts8 = max(1, n_mem8//7); tv8 = list(range(0, n_mem8+1, ts8))
    if n_mem8 not in tv8: tv8.append(n_mem8)
    cb8b.set_ticks(tv8); cb8b.set_ticklabels([str(i) for i in tv8])
    cb8b.set_label(f"Members showing degradation (/{n_mem8})", fontsize=7, labelpad=2)
    cb8b.ax.tick_params(labelsize=6.5, width=0.3, length=1.5); cb8b.outline.set_linewidth(0.3)

    # ══ (c) Stream-order summary ════════════════════════════════
    ax8c = fig8.add_axes([0.76, 0.25, 0.22, 0.53])

    gdf8["so"] = gdf8["StreamOrde"].astype(int)
    sos = range(1, 8)
    yb = np.arange(7)

    for j, s in enumerate(sos):
        dr = gdf8.loc[gdf8["so"] == s, "delta_red"].values * 100
        dr = dr[np.isfinite(dr)]
        if len(dr) < 3: continue
        md = np.median(dr)
        q25 = np.percentile(dr, 25)
        q75 = np.percentile(dr, 75)
        ax8c.barh(yb[j], q75-q25, left=q25, height=0.55,
                  color="#e08214", alpha=0.45, edgecolor="none")
        ax8c.plot(md, yb[j], "o", color="#e08214", ms=4.5,
                  markeredgecolor="white", markeredgewidth=0.3, zorder=5)

    ax8c.axvline(0, color="#aaa", ls="--", lw=0.6, zorder=0)
    ax8c.set_yticks(yb)
    ax8c.set_xlim(-5, 5)
    ax8c.set_yticklabels([str(s) for s in sos])
    ax8c.set_ylabel("Stream order", fontsize=7)
    ax8c.set_xlabel(r"$\Delta$ peak reduction (pp)", fontsize=7)
    ax8c.tick_params(labelsize=6.5)
    lgrid(ax8c, alpha=0.08)
    plabel(ax8c, "c", x=-0.14, y=1.04)

    for ext in ("png","pdf"): fig8.savefig(FIG_DIR / f"fig8_effectiveness.{ext}")
    plt.show()
    print(f"  ✓ Saved fig8_effectiveness")